# Demeter Filter Experimentation: Optimizing the FFT+HSV SVM Pipeline
In this notebook, we explore alternative windowing and filtering techniques applied to the 2D Fast Fourier Transform (FFT) magnitude spectrum.

Our goal is to see if different filter types (like Hamming, Hanning, or Blackman windows) can improve the feature extraction over our baseline Gaussian-fade windowing, ultimately boosting the Macro F1-Score of our lightweight SVM model.

In [ ]:
import os
import sys
import cv2
import numpy as np
import matplotlib.pyplot as plt
import scipy.signal as signal
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import train_test_split

# Add src to path
sys.path.append(os.path.abspath('..'))

## 1. Define Windowing Functions
We implement a standard 2D window generator that can create Gaussian, Hamming, Hanning, and Blackman windows to smooth the spatial domain before FFT, minimizing high-frequency boundary spikes.

In [ ]:
def create_2d_window(size=64, window_type='gaussian', std=15):
    """Generates a 2D windowing mask."""
    if window_type == 'gaussian':
        1d_win = signal.windows.gaussian(size, std=std)
    elif window_type == 'hamming':
        1d_win = signal.windows.hamming(size)
    elif window_type == 'hanning':
        1d_win = signal.windows.hann(size)
    elif window_type == 'blackman':
        1d_win = signal.windows.blackman(size)
    else:
        return np.ones((size, size))
        
    # Create 2D window by outer product
    2d_win = np.outer(1d_win, 1d_win)
    return 2d_win

# Visualize the windows
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
windows = ['gaussian', 'hamming', 'hanning', 'blackman']
for ax, w in zip(axes, windows):
    win_mask = create_2d_window(window_type=w)
    im = ax.imshow(win_mask, cmap='viridis')
    ax.set_title(f"{w.capitalize()} Window")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

## 2. Extraction Pipeline with Dynamic Windowing
We modify our core feature extraction to apply a chosen window to the segmented grayscale leaf before performing the 2D FFT.

In [ ]:
def extract_features_with_window(img_path, window_mask):
    img_bgr = cv2.imread(img_path)
    if img_bgr is None: return None, None
    
    IMG_SIZE = 64
    img_bgr = cv2.resize(img_bgr, (IMG_SIZE, IMG_SIZE))
    
    # Segmentation
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    _, mask = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    
    # Apply Segmentation
    segmented = cv2.bitwise_and(gray, gray, mask=mask)
    
    # APPLY WINDOW (Tapering)
    tapered = segmented * window_mask
    
    # 2D FFT on tapered image
    fft_img = np.fft.fft2(tapered)
    mag_spectrum = 20 * np.log(np.abs(np.fft.fftshift(fft_img)) + 1).flatten()
    
    # HSV Histogram (unchanged)
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    h_hist = cv2.calcHist([hsv], [0], mask, [32], [0, 180]).flatten()
    s_hist = cv2.calcHist([hsv], [1], mask, [32], [0, 256]).flatten()
    color_hist = np.concatenate([h_hist, s_hist])
    
    return mag_spectrum, color_hist

## 3. Train and Evaluate SVM per Window Type
Here you would load the full dataset, extract features using each window type, train an SVM, and compare the F1 Scores to find the optimal filter.

In [ ]:
print("Workflow to execute:")
print("1. Load Dataset")
print("2. For each window type in ['none', 'gaussian', 'hamming', 'hanning', 'blackman']:")
print("     a. Extract features (mag_spectrum + color_hist)")
print("     b. Fit PCA (n=100) on FFT features")
print("     c. Scale all features")
print("     d. Train SVC(kernel='rbf')")
print("     e. Calculate and log Macro F1-Score")
print("3. Compare results to find the highest F1-Score.")
# Note: Full dataset processing is omitted here for brevity, but this script provides the framework.